
# Diagnose prior inner-boundary condition distribution for fixed-annulus SNO

This notebook checks whether the **target inner-boundary condition** used in inference is inside, or far from, the training prior distribution.

Current purpose:

1. Use the same PI-sampler prior frequency set as training:
   \[
   \sigma \in \{3,5,7\}.
   \]
2. Generate prior samples and extract the induced inner-boundary flux/token:
   \[
   g_{\mathrm{prior}}(\theta).
   \]
3. Compare the prior boundary flux distribution with:
   \[
   g_{+}(\theta)=\cos\theta,
   \qquad
   g_{-}(\theta)=-\cos\theta.
   \]
4. Export all curves and diagnostic metrics to `.mat` for MATLAB visualization.

The comparison with both `+cos(theta)` and `-cos(theta)` is intentional. It helps diagnose both distribution shift and possible inner-normal sign convention mismatch.


In [ ]:

# ============================================================
# 0. Basic settings
# ============================================================

import os
import sys
from pathlib import Path
import copy

import numpy as np
import jax
import jax.numpy as jnp
from scipy.io import savemat
import matplotlib.pyplot as plt

# Optional: set GPU before importing JAX in a fresh kernel.
# If JAX has already been imported, changing CUDA_VISIBLE_DEVICES here may not take effect.
# os.environ["CUDA_VISIBLE_DEVICES"] = "5"
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

print("JAX devices:", jax.devices())



## 1. Configure project path and sampling parameters

Modify `PROJECT_DIR`, `OUT_DIR`, and `RUN_NAME` to match your fixed-annulus SNO code/output location.

This notebook does **not** load FE or SNO parameters. It only calls the PI-sampler through `sample_batch` and reads `batch.boundary_flux`.


In [ ]:

# ============================================================
# 1. User configuration
# ============================================================

# Directory containing config.py, data.py, models.py, train.py
PROJECT_DIR = Path("/home/user/data/Hollon/海洋工程水动力/annulus_sno_annulus_only_v2/annulus_sno_annulus_only_v2")

# Output directory and run name, consistent with your existing experiment
OUT_DIR = Path("/home/user/data/Hollon/海洋工程水动力/annulus_sno_annulus_only_v2/out")
RUN_NAME = "test"

# Prior sigmas used in training distribution
SIGMA_LIST = (3.0, 5.0, 7.0)

# Number of samples generated for each sigma.
# Total samples = len(SIGMA_LIST) * N_PER_SIGMA.
N_PER_SIGMA = 512

# Geometry and PDE parameters should match training.
R_INNER = 0.2
R_OUTER = 1.0
K_MIN = 1.0
K_MAX = 1.0

# Discretization should match training.
THETA_SIZE = 128
RADIAL_SIZE = 32
RANDOM_PROBE_POINTS = 1024
HIDDEN_BNN = 256

# Random seed for reproducible diagnostics
SEED = 20260624

# Export file name
MAT_NAME = "prior_boundary_flux_distribution_sigma_3_5_7.mat"

assert PROJECT_DIR.exists(), f"PROJECT_DIR does not exist: {PROJECT_DIR}"
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR =", PROJECT_DIR)
print("Export dir  =", OUT_DIR / RUN_NAME)


In [ ]:

# ============================================================
# 2. Import fixed-annulus SNO modules
# ============================================================

from config import AnnulusConfig
from data import sample_batch, make_theta, inner_boundary_coords

cfg_base = AnnulusConfig()

# Force the diagnostic config to match your trained fixed-annulus setting.
cfg_base.out_dir = str(OUT_DIR)
cfg_base.run_name = RUN_NAME

cfg_base.r_inner = R_INNER
cfg_base.r_outer = R_OUTER
cfg_base.k_min = K_MIN
cfg_base.k_max = K_MAX

cfg_base.theta_size = THETA_SIZE
cfg_base.radial_size = RADIAL_SIZE
cfg_base.random_probe_points = RANDOM_PROBE_POINTS
cfg_base.hidden_bnn = HIDDEN_BNN

# For this diagnostic, we generate one exact sigma at a time.
cfg_base.num_repeats = 1
cfg_base.sample_size = N_PER_SIGMA

print(cfg_base)
print("Output directory:", cfg_base.output_dir)



## 2. Generate prior boundary flux samples

`sample_batch` returns `batch.boundary_flux`, which is exactly the inner-boundary condition token used by the Transformer training data:

\[
\texttt{condition token}= [x_b, y_b, g_{\mathrm{prior}}(\theta)].
\]

To keep sigma labels exact, this notebook generates each sigma separately by temporarily setting:

```python
cfg.sigma_list = (sigma,)
```

Then the resulting mixture has the same sigma support as training, while still retaining per-sigma labels for diagnostics.


In [ ]:

# ============================================================
# 3. Generate boundary flux samples for each sigma
# ============================================================

def to_numpy(x):
    return np.asarray(jax.device_get(x))

key = jax.random.PRNGKey(SEED)

boundary_flux_by_sigma = []   # list of [N_PER_SIGMA, Nt]
boundary_coords_by_sigma = [] # list of [N_PER_SIGMA, Nt, 2]
k_values_by_sigma = []        # list of [N_PER_SIGMA]

for sigma in SIGMA_LIST:
    cfg = copy.deepcopy(cfg_base)
    cfg.sigma_list = (float(sigma),)
    cfg.num_repeats = 1
    cfg.sample_size = N_PER_SIGMA

    key, subkey = jax.random.split(key)
    batch = sample_batch(subkey, cfg)

    g = to_numpy(batch.boundary_flux)          # [B, Nt]
    xb = to_numpy(batch.boundary_coords)       # [B, Nt, 2]
    kv = to_numpy(batch.k_values)              # [B]

    boundary_flux_by_sigma.append(g)
    boundary_coords_by_sigma.append(xb)
    k_values_by_sigma.append(kv)

    print(f"sigma={sigma:4.1f} | boundary_flux shape = {g.shape} | mean={g.mean():+.3e}, std={g.std():.3e}")

boundary_flux_by_sigma = np.stack(boundary_flux_by_sigma, axis=0)       # [Nsigma, B, Nt]
boundary_coords_by_sigma = np.stack(boundary_coords_by_sigma, axis=0)   # [Nsigma, B, Nt, 2]
k_values_by_sigma = np.stack(k_values_by_sigma, axis=0)                 # [Nsigma, B]

sigma_values = np.asarray(SIGMA_LIST, dtype=np.float64)

print("boundary_flux_by_sigma:", boundary_flux_by_sigma.shape)
print("boundary_coords_by_sigma:", boundary_coords_by_sigma.shape)
print("k_values_by_sigma:", k_values_by_sigma.shape)


In [ ]:

# ============================================================
# 4. Build theta and target boundary conditions
# ============================================================

# Use the same theta convention as the SNO code.
theta = np.linspace(0.0, 2.0 * np.pi, THETA_SIZE, endpoint=False).astype(np.float64)  # [Nt]

# Two candidate target fluxes for sign-convention diagnosis.
target_flux_cos = np.cos(theta)       # +cos(theta)
target_flux_neg_cos = -np.cos(theta)  # -cos(theta)
target_flux_sin = np.sin(theta)

# Inner boundary coordinates are the same for all samples in fixed annulus.
# Use the first sigma / first sample for export convenience.
boundary_coords = boundary_coords_by_sigma[0, 0]  # [Nt, 2]

print("theta shape:", theta.shape)
print("target_flux_cos shape:", target_flux_cos.shape)
print("boundary_coords shape:", boundary_coords.shape)



## 3. Compute distribution diagnostics

For each prior sample \(g_i(\theta)\), the notebook computes:

- relative distance to \(+\cos\theta\):
  \[
  \frac{\|g_i-\cos\theta\|_2}{\|\cos\theta\|_2};
  \]
- relative distance to \(-\cos\theta\):
  \[
  \frac{\|g_i+\cos\theta\|_2}{\|\cos\theta\|_2};
  \]
- correlation with \(+\cos\theta\) and \(-\cos\theta\);
- projection coefficients onto \(1,\cos\theta,\sin\theta\);
- residual after first-harmonic fitting;
- Fourier energy spectrum and low-mode energy ratios.


In [ ]:

# ============================================================
# 5. Diagnostic functions
# ============================================================

def rel_l2(a, b, axis=-1, eps=1e-12):
    return np.linalg.norm(a - b, axis=axis) / (np.linalg.norm(b, axis=axis) + eps)


def centered_corr(a, b, axis=-1, eps=1e-12):
    a0 = a - np.mean(a, axis=axis, keepdims=True)
    b0 = b - np.mean(b, axis=axis, keepdims=True)
    num = np.sum(a0 * b0, axis=axis)
    den = np.sqrt(np.sum(a0**2, axis=axis) * np.sum(b0**2, axis=axis)) + eps
    return num / den


def projection_coeff(g, basis):
    # g: [..., Nt], basis: [Nt]
    return np.sum(g * basis[None, :], axis=-1) / (np.sum(basis**2) + 1e-12)


def first_harmonic_fit_metrics(g):
    # Fit g(theta) approximately as a0 + a_cos*cos(theta) + a_sin*sin(theta).
    a0 = np.mean(g, axis=-1)
    a_cos = projection_coeff(g, target_flux_cos)
    a_sin = projection_coeff(g, target_flux_sin)

    g_fit = (
        a0[..., None]
        + a_cos[..., None] * target_flux_cos[None, :]
        + a_sin[..., None] * target_flux_sin[None, :]
    )

    rel_residual = np.linalg.norm(g - g_fit, axis=-1) / (np.linalg.norm(g, axis=-1) + 1e-12)
    first_harmonic_amplitude = np.sqrt(a_cos**2 + a_sin**2)
    first_harmonic_phase = np.arctan2(a_sin, a_cos)

    return a0, a_cos, a_sin, first_harmonic_amplitude, first_harmonic_phase, rel_residual, g_fit


def rfft_energy(g):
    # Compute real FFT energy for g(theta). Input shape: [Nsigma, B, Nt].
    coeff = np.fft.rfft(g, axis=-1) / g.shape[-1]
    energy = np.abs(coeff)**2
    return coeff, energy


def low_mode_ratio(energy, max_mode):
    total = np.sum(energy, axis=-1) + 1e-12
    part = np.sum(energy[..., :max_mode+1], axis=-1)
    return part / total


In [ ]:

# ============================================================
# 6. Compute sample-wise diagnostics
# ============================================================

G = boundary_flux_by_sigma  # [Nsigma, B, Nt]
Nsigma, B, Nt = G.shape
G_flat = G.reshape(Nsigma * B, Nt)

# Relative distances to target candidates
err_to_cos = rel_l2(G.reshape(-1, Nt), target_flux_cos[None, :]).reshape(Nsigma, B)
err_to_neg_cos = rel_l2(G.reshape(-1, Nt), target_flux_neg_cos[None, :]).reshape(Nsigma, B)

# Correlations
corr_to_cos = centered_corr(G.reshape(-1, Nt), target_flux_cos[None, :]).reshape(Nsigma, B)
corr_to_neg_cos = centered_corr(G.reshape(-1, Nt), target_flux_neg_cos[None, :]).reshape(Nsigma, B)

# First harmonic fit
(
    flux_mean_mode0,
    flux_coeff_cos,
    flux_coeff_sin,
    flux_first_harmonic_amp,
    flux_first_harmonic_phase,
    flux_first_harmonic_rel_residual,
    flux_first_harmonic_fit_flat,
) = first_harmonic_fit_metrics(G_flat)

flux_mean_mode0 = flux_mean_mode0.reshape(Nsigma, B)
flux_coeff_cos = flux_coeff_cos.reshape(Nsigma, B)
flux_coeff_sin = flux_coeff_sin.reshape(Nsigma, B)
flux_first_harmonic_amp = flux_first_harmonic_amp.reshape(Nsigma, B)
flux_first_harmonic_phase = flux_first_harmonic_phase.reshape(Nsigma, B)
flux_first_harmonic_rel_residual = flux_first_harmonic_rel_residual.reshape(Nsigma, B)
flux_first_harmonic_fit = flux_first_harmonic_fit_flat.reshape(Nsigma, B, Nt)

# FFT spectrum
fft_coeff, fft_energy = rfft_energy(G)
fft_freq_index = np.arange(fft_energy.shape[-1], dtype=np.int64)
energy_ratio_le_1 = low_mode_ratio(fft_energy, max_mode=1)
energy_ratio_le_3 = low_mode_ratio(fft_energy, max_mode=3)
energy_ratio_le_5 = low_mode_ratio(fft_energy, max_mode=5)

print("err_to_cos shape:", err_to_cos.shape)
print("fft_energy shape:", fft_energy.shape)

# Text summary
for i, sigma in enumerate(SIGMA_LIST):
    print("\n" + "="*72)
    print(f"sigma = {sigma}")
    print(f"relL2 to +cos: mean={err_to_cos[i].mean():.4e}, median={np.median(err_to_cos[i]):.4e}, min={err_to_cos[i].min():.4e}")
    print(f"relL2 to -cos: mean={err_to_neg_cos[i].mean():.4e}, median={np.median(err_to_neg_cos[i]):.4e}, min={err_to_neg_cos[i].min():.4e}")
    print(f"corr  to +cos: mean={corr_to_cos[i].mean():+.4f}, max={corr_to_cos[i].max():+.4f}")
    print(f"corr  to -cos: mean={corr_to_neg_cos[i].mean():+.4f}, max={corr_to_neg_cos[i].max():+.4f}")
    print(f"first harmonic residual: mean={flux_first_harmonic_rel_residual[i].mean():.4e}")
    print(f"energy ratio <=1: mean={energy_ratio_le_1[i].mean():.4f}")
    print(f"energy ratio <=3: mean={energy_ratio_le_3[i].mean():.4f}")


In [ ]:

# ============================================================
# 7. Distribution envelopes over theta
# ============================================================

# Per-sigma theta-wise statistics: [Nsigma, Nt]
flux_mean_theta = np.mean(G, axis=1)
flux_std_theta = np.std(G, axis=1)
flux_p01_theta = np.percentile(G, 1, axis=1)
flux_p05_theta = np.percentile(G, 5, axis=1)
flux_p50_theta = np.percentile(G, 50, axis=1)
flux_p95_theta = np.percentile(G, 95, axis=1)
flux_p99_theta = np.percentile(G, 99, axis=1)
flux_min_theta = np.min(G, axis=1)
flux_max_theta = np.max(G, axis=1)

# Mixture distribution over all sigmas and all samples: [Nt]
G_mix = G.reshape(Nsigma * B, Nt)
flux_mix_mean_theta = np.mean(G_mix, axis=0)
flux_mix_std_theta = np.std(G_mix, axis=0)
flux_mix_p01_theta = np.percentile(G_mix, 1, axis=0)
flux_mix_p05_theta = np.percentile(G_mix, 5, axis=0)
flux_mix_p50_theta = np.percentile(G_mix, 50, axis=0)
flux_mix_p95_theta = np.percentile(G_mix, 95, axis=0)
flux_mix_p99_theta = np.percentile(G_mix, 99, axis=0)
flux_mix_min_theta = np.min(G_mix, axis=0)
flux_mix_max_theta = np.max(G_mix, axis=0)

print("flux_mean_theta:", flux_mean_theta.shape)
print("flux_mix_mean_theta:", flux_mix_mean_theta.shape)



## 4. Quick visualization inside notebook

These plots are only quick checks. The exported `.mat` file contains the complete data for high-quality MATLAB visualization.


In [ ]:

# ============================================================
# 8. Quick plot: mixture envelope vs target cos(theta)
# ============================================================

plt.figure(figsize=(10, 4.5))
plt.fill_between(theta, flux_mix_p05_theta, flux_mix_p95_theta, alpha=0.25, label='prior 5%-95%')
plt.plot(theta, flux_mix_p50_theta, linewidth=2, label='prior median')
plt.plot(theta, target_flux_cos, '--', linewidth=2, label=r'$+\cos\theta$')
plt.plot(theta, target_flux_neg_cos, ':', linewidth=2, label=r'$-\cos\theta$')
plt.xlabel(r'$\theta$')
plt.ylabel(r'$g(\theta)$')
plt.title('Mixture prior boundary flux distribution')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# 9. Quick plot: error and correlation distributions
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].boxplot([err_to_cos[i] for i in range(Nsigma)], labels=[str(s) for s in SIGMA_LIST])
axes[0].set_title(r'Relative $L^2$ distance to $+\cos\theta$')
axes[0].set_xlabel(r'$\sigma$')
axes[0].set_ylabel('relative L2')
axes[0].grid(True, alpha=0.3)

axes[1].boxplot([err_to_neg_cos[i] for i in range(Nsigma)], labels=[str(s) for s in SIGMA_LIST])
axes[1].set_title(r'Relative $L^2$ distance to $-\cos\theta$')
axes[1].set_xlabel(r'$\sigma$')
axes[1].grid(True, alpha=0.3)

axes[2].boxplot([corr_to_cos[i] for i in range(Nsigma)], labels=[str(s) for s in SIGMA_LIST])
axes[2].set_title(r'Correlation with $+\cos\theta$')
axes[2].set_xlabel(r'$\sigma$')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# 10. Quick plot: mean Fourier energy spectrum
# ============================================================

mean_fft_energy_by_sigma = np.mean(fft_energy, axis=1)  # [Nsigma, Nfreq]
mean_fft_energy_mix = np.mean(fft_energy.reshape(Nsigma * B, -1), axis=0)

max_mode_to_plot = min(30, mean_fft_energy_mix.shape[0] - 1)
mode_ids = np.arange(max_mode_to_plot + 1)

plt.figure(figsize=(9, 4.5))
for i, sigma in enumerate(SIGMA_LIST):
    plt.semilogy(mode_ids, mean_fft_energy_by_sigma[i, :max_mode_to_plot+1] + 1e-20, marker='o', label=f'sigma={sigma}')
plt.xlabel('Fourier mode index')
plt.ylabel('mean spectral energy')
plt.title('Prior boundary flux Fourier energy spectrum')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()



## 5. Export `.mat` file

The exported file contains both raw boundary flux samples and processed statistics.

Important variables:

- `boundary_flux_by_sigma`: `[Nsigma, N_per_sigma, theta_size]`
- `sigma_values`: `[Nsigma, 1]`
- `theta`: `[theta_size, 1]`
- `target_flux_cos`: `+cos(theta)`
- `target_flux_neg_cos`: `-cos(theta)`
- `err_to_cos`, `err_to_neg_cos`: sample-wise relative distances
- `corr_to_cos`, `corr_to_neg_cos`: sample-wise correlations
- `fft_energy`: sample-wise Fourier energy
- `flux_*_theta`: theta-wise distribution envelopes


In [ ]:

# ============================================================
# 11. Export to MATLAB .mat file
# ============================================================

export_dir = cfg_base.output_dir
export_dir.mkdir(parents=True, exist_ok=True)
mat_path = export_dir / MAT_NAME

mat_data = {
    # Metadata
    'sigma_values': sigma_values[:, None],
    'n_per_sigma': np.array([[N_PER_SIGMA]], dtype=np.int64),
    'theta_size': np.array([[THETA_SIZE]], dtype=np.int64),
    'r_inner': np.array([[R_INNER]], dtype=np.float64),
    'r_outer': np.array([[R_OUTER]], dtype=np.float64),
    'k_min': np.array([[K_MIN]], dtype=np.float64),
    'k_max': np.array([[K_MAX]], dtype=np.float64),
    'seed': np.array([[SEED]], dtype=np.int64),

    # Coordinates and target fluxes
    'theta': theta[:, None],
    'boundary_coords': boundary_coords,
    'boundary_coords_by_sigma': boundary_coords_by_sigma,
    'target_flux_cos': target_flux_cos[None, :],
    'target_flux_neg_cos': target_flux_neg_cos[None, :],
    'target_flux_sin': target_flux_sin[None, :],

    # Raw prior samples
    'boundary_flux_by_sigma': boundary_flux_by_sigma,
    'k_values_by_sigma': k_values_by_sigma,

    # Sample-wise distance/correlation diagnostics
    'err_to_cos': err_to_cos,
    'err_to_neg_cos': err_to_neg_cos,
    'corr_to_cos': corr_to_cos,
    'corr_to_neg_cos': corr_to_neg_cos,

    # First harmonic fit diagnostics
    'flux_mean_mode0': flux_mean_mode0,
    'flux_coeff_cos': flux_coeff_cos,
    'flux_coeff_sin': flux_coeff_sin,
    'flux_first_harmonic_amp': flux_first_harmonic_amp,
    'flux_first_harmonic_phase': flux_first_harmonic_phase,
    'flux_first_harmonic_rel_residual': flux_first_harmonic_rel_residual,
    'flux_first_harmonic_fit': flux_first_harmonic_fit,

    # Fourier diagnostics
    'fft_freq_index': fft_freq_index[:, None],
    'fft_coeff_real': np.real(fft_coeff),
    'fft_coeff_imag': np.imag(fft_coeff),
    'fft_energy': fft_energy,
    'mean_fft_energy_by_sigma': mean_fft_energy_by_sigma,
    'mean_fft_energy_mix': mean_fft_energy_mix[None, :],
    'energy_ratio_le_1': energy_ratio_le_1,
    'energy_ratio_le_3': energy_ratio_le_3,
    'energy_ratio_le_5': energy_ratio_le_5,

    # Per-sigma theta-wise statistics
    'flux_mean_theta': flux_mean_theta,
    'flux_std_theta': flux_std_theta,
    'flux_p01_theta': flux_p01_theta,
    'flux_p05_theta': flux_p05_theta,
    'flux_p50_theta': flux_p50_theta,
    'flux_p95_theta': flux_p95_theta,
    'flux_p99_theta': flux_p99_theta,
    'flux_min_theta': flux_min_theta,
    'flux_max_theta': flux_max_theta,

    # Mixture theta-wise statistics
    'flux_mix_mean_theta': flux_mix_mean_theta[None, :],
    'flux_mix_std_theta': flux_mix_std_theta[None, :],
    'flux_mix_p01_theta': flux_mix_p01_theta[None, :],
    'flux_mix_p05_theta': flux_mix_p05_theta[None, :],
    'flux_mix_p50_theta': flux_mix_p50_theta[None, :],
    'flux_mix_p95_theta': flux_mix_p95_theta[None, :],
    'flux_mix_p99_theta': flux_mix_p99_theta[None, :],
    'flux_mix_min_theta': flux_mix_min_theta[None, :],
    'flux_mix_max_theta': flux_mix_max_theta[None, :],
}

savemat(mat_path, mat_data, do_compression=True)

print('Saved MATLAB file to:')
print(mat_path)


In [ ]:

# ============================================================
# 12. Compact summary table for notebook inspection
# ============================================================

try:
    import pandas as pd

    rows = []
    for i, sigma in enumerate(SIGMA_LIST):
        rows.append({
            'sigma': sigma,
            'relL2_to_+cos_mean': float(err_to_cos[i].mean()),
            'relL2_to_+cos_median': float(np.median(err_to_cos[i])),
            'relL2_to_+cos_min': float(err_to_cos[i].min()),
            'relL2_to_-cos_mean': float(err_to_neg_cos[i].mean()),
            'relL2_to_-cos_median': float(np.median(err_to_neg_cos[i])),
            'relL2_to_-cos_min': float(err_to_neg_cos[i].min()),
            'corr_to_+cos_mean': float(corr_to_cos[i].mean()),
            'corr_to_+cos_max': float(corr_to_cos[i].max()),
            'first_harmonic_residual_mean': float(flux_first_harmonic_rel_residual[i].mean()),
            'energy_ratio_le_1_mean': float(energy_ratio_le_1[i].mean()),
            'energy_ratio_le_3_mean': float(energy_ratio_le_3[i].mean()),
        })

    df_summary = pd.DataFrame(rows)
    display(df_summary)
except Exception as e:
    print('Pandas summary skipped:', e)
